# 그래프생성_함수설명

각 설명 다음의 코드 셀을 Shift+Enter로 실행하세요. 이 노트북은 함수를 정의하고 내부 모듈을 준비합니다. 실제 전처리·학습 호출은 **00번 통합 노트북**에서 합니다. 내부 파일은 셀에서 자동 생성하므로 별도 Python 파일을 준비할 필요가 없습니다.

In [ ]:
from pathlib import Path
import sys, json
# 통합 노트북은 이 폴더를 작업 폴더로 지정합니다. 개별 실행도 같은 폴더에서 하세요.
WORKSPACE = Path.cwd().resolve()
RUNTIME = WORKSPACE / '_runtime'
(RUNTIME / 'gnn_pipeline').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'gnn_pipeline' / '__init__.py').touch()
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


## 실행 셀을 내부 모듈로 연결

아래 함수는 이 노트북의 코드 셀을 그대로 저장해 다음 단계에서 import할 수 있게 합니다.

In [ ]:
def export_cells(notebook_name, module_name):
    """이 노트북의 실제 코드 셀을 모아 내부 import용 모듈을 생성합니다."""
    notebook = json.loads((WORKSPACE / notebook_name).read_text(encoding='utf-8'))
    blocks = [cell['source'] for cell in notebook['cells']
              if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('module') == module_name]
    source = '\n\n'.join(''.join(block) if isinstance(block, list) else block for block in blocks) + '\n'
    path = RUNTIME / 'gnn_pipeline' / (module_name + '.py')
    compile(source, str(path), 'exec')
    path.write_text(source, encoding='utf-8')
    print('내부 모듈 준비:', path.name)
    return path


## 라이브러리·상수 준비

다음 함수가 사용하는 라이브러리와 설정을 준비합니다.

In [ ]:
"""Ordered preprocessing, graph construction and validation-only training."""
from dataclasses import asdict, replace
from pathlib import Path
import hashlib
import json
import os
import numpy as np
import pandas as pd
from gnn_pipeline import data

SCHEMA = {'node_features': ['log_out_count', 'log_in_count', 'log_out_neighbors', 'log_in_neighbors', 'log_self_count', 'log_reciprocal_neighbors', 'mean_paid_currency_z', 'paid_outlier_fraction', 'log_currency_count', 'log_active_days', 'type_Corporation', 'type_Partnership', 'type_Sole Proprietorship', 'type_Individual', 'type_Country', 'type_Direct', 'log_entity_observed_accounts', 'log_entity_out_diversity', 'log_entity_in_diversity'], 'edge_features': ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'sender_gap_minutes', 'sender_count_1h', 'receiver_gap_minutes', 'receiver_count_1h', 'paid_currency_z', 'pair_history_count', 'reverse_history_count', 'pair_age_at_day_start', 'same_entity', 'same_known_country', 'both_countries_known', 'same_currency', 'is_self_transfer', 'amt_z_pay_ccy']}



## `load_settings`

설정 JSON을 읽습니다. 상대 경로는 설정 파일이 있는 폴더 기준으로 계산하므로 컴퓨터가 바뀌어도 사용할 수 있습니다.

**입력:** `path`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `(settings, cfg)`

In [ ]:
def load_settings(path):
    """Resolve relative paths from config.json, never from the original machine."""
    path = Path(path).resolve()
    settings = json.loads(path.read_text())
    for key in ('source', 'output'):
        value = Path(settings['data'][key]).expanduser()
        settings['data'][key] = str(value if value.is_absolute() else (path.parent / value).resolve())
    cfg = data.Config(**settings['data'])
    cfg.validate()
    if settings['sampling'] not in ('original', 'random_under', 'cluster_under'):
        raise ValueError('Unknown sampling method')
    if not 0 < settings['negative_rate'] <= 1 or settings['clusters'] < 1:
        raise ValueError('Invalid sampling settings')
    from gnn_pipeline.models import MODEL_NAMES
    if not settings['models'] or not set(settings['models']) <= set(MODEL_NAMES):
        raise ValueError('Invalid model names')
    if not settings['seeds'] or len(set(settings['seeds'])) != len(settings['seeds']):
        raise ValueError('Seeds must be nonempty and unique')
    return settings, cfg

## `bind_run`

설정·피처·코드의 지문을 저장합니다. 변경된 조건으로 기존 결과를 잘못 재사용하면 중단합니다.

**입력:** `settings`, `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `root`

In [ ]:
def bind_run(settings, cfg):
    """Prevent changed settings/code from silently reusing cached results."""
    root = Path(cfg.output).parent
    root.mkdir(parents=True, exist_ok=True)
    package = Path(__file__).parent
    identity = {'settings': settings, 'schema': SCHEMA,
                'code': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(package.glob('*.py'))}}
    path = root / 'pipeline_manifest.json'
    if path.exists() and json.loads(path.read_text()) != identity:
        raise ValueError('Settings/code changed. Choose a new data.output parent directory.')
    data.json_write(path, identity)
    return root

## `preprocess`

데이터 정제, 과거 이력, 피처 생성을 순서대로 호출합니다. 완료된 단계는 설정과 원본 파일이 일치할 때 재사용합니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `pd.read_csv(Path(cfg.output) / 'dataset_summary.csv')`

In [ ]:
def preprocess(cfg):
    """Validate raw files, normalize transactions, then build history and features."""
    stages = [('dataset', data.build_dataset), ('pairagg', data.build_pairagg), ('features', data.build_features)]
    for name, action in stages:
        if (Path(cfg.output) / f'{name}.json').exists():
            data.require_stage(cfg, name)
            print('Reusing checked stage:', name, flush=True)
        else:
            print('Building:', name, flush=True)
            action(cfg)
    return pd.read_csv(Path(cfg.output) / 'dataset_summary.csv')

## `amount_z`

학습 이전 기간에서 구한 통화별 평균과 표준편차로 지불 금액을 표준화합니다. 거래 ID를 맞춰 문맥·대상 거래에 연결합니다.

**입력:** `cfg`, `day`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `(z[np.searchsorted(raw.edge_id, context.edge_id)], z[np.searchsorted(raw.edge_id, day.target_ids)])`

In [ ]:
def amount_z(cfg, day):
    """Join context/target IDs to paid-amount z-scores fitted before training."""
    connection = data.connect(cfg, read_only=True)
    try:
        start = str((pd.Timestamp(day.day) - pd.Timedelta(days=cfg.history_days - 1)).date())
        context = connection.execute(f'''SELECT edge_id, src, dst FROM tx
            WHERE day >= {data.sql_string(start)}::DATE AND day <= {data.sql_string(day.day)}::DATE
            QUALIFY row_number() OVER(PARTITION BY dst ORDER BY ts DESC, edge_id DESC) <= {cfg.incoming_neighbors}
            ORDER BY dst, ts DESC, edge_id DESC''').fetchdf()
        np.testing.assert_array_equal(context.src, day.context_src)
        np.testing.assert_array_equal(context.dst, day.context_dst)
        connection.register('wanted', pd.DataFrame({'edge_id': np.union1d(context.edge_id, day.target_ids)}))
        raw = connection.execute('''SELECT t.edge_id, ln(1+t.paid::DOUBLE) AS amount, t.payment_currency
                                   FROM tx t JOIN wanted USING(edge_id) ORDER BY edge_id''').fetchdf()
        stats = connection.execute(f'''SELECT payment_currency, avg(ln(1+paid::DOUBLE)) AS m,
            stddev_pop(ln(1+paid::DOUBLE)) AS sd FROM tx
            WHERE day >= {data.sql_string(cfg.calibration_start)}::DATE AND day < {data.sql_string(cfg.train_start)}::DATE
            GROUP BY payment_currency''').fetchdf().set_index('payment_currency')
        aligned = stats.reindex(raw.payment_currency)
        mean, sd = aligned.m.to_numpy(), aligned.sd.to_numpy()
        z = np.where(np.isfinite(mean) & np.isfinite(sd),
                     np.clip((raw.amount.to_numpy() - mean) / (sd + 1e-6), -10, 10), 0.).astype('float32')
        data.finite('paid amount z', z)
        return z[np.searchsorted(raw.edge_id, context.edge_id)], z[np.searchsorted(raw.edge_id, day.target_ids)]
    finally:
        connection.close()

## `graph36`

원래 그래프에서 계좌 특징 19개와 거래 특징 16개를 선택하고 금액 특징 1개를 추가합니다. 결과는 19+17 피처입니다.

**입력:** `cfg`, `date`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `replace(day, node_x=day.node_x[:, nodes], context_x=np.column_stack([day.context_x[:, edges], context_z]), target_x=np.column_stack([day.target_x[:, edges], target_z]))`

In [ ]:
def graph36(cfg, date):
    """Build a daily graph and select 19 node plus 17 transaction features."""
    day = data.load_day(cfg, date, with_labels=True)
    context_z, target_z = amount_z(cfg, day)
    nodes = [data.NODE_FEATURES.index(name) for name in SCHEMA['node_features']]
    edges = [data.EDGE_FEATURES.index(name) for name in SCHEMA['edge_features'][:-1]]
    return replace(day, node_x=day.node_x[:, nodes],
                   context_x=np.column_stack([day.context_x[:, edges], context_z]),
                   target_x=np.column_stack([day.target_x[:, edges], target_z]))

## `graph_dates`

실제 거래가 존재하는 학습/검증 날짜만 읽습니다. 테스트 구간은 처리하지 않습니다.

**입력:** `cfg`, `split`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `[row[0] for row in rows]`

In [ ]:
def graph_dates(cfg, split):
    """Read observed dates in the requested train/validation interval."""
    if split not in ('train', 'val'):
        raise ValueError('This pipeline does not evaluate test labels')
    start, end = (cfg.train_start, cfg.val_start) if split == 'train' else (cfg.val_start, cfg.test_start)
    connection = data.connect(cfg, read_only=True)
    try:
        rows = connection.execute(f"SELECT DISTINCT strftime(day, '%Y-%m-%d') FROM tx WHERE day >= {data.sql_string(start)}::DATE AND day < {data.sql_string(end)}::DATE ORDER BY 1").fetchall()
        return [row[0] for row in rows]
    finally:
        connection.close()

## `build_graphs`

학습·검증 날짜별 그래프를 만들고 배열로 저장합니다. 다음 단계는 이 파일을 그대로 읽습니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `frame`

In [ ]:
def build_graphs(cfg):
    """Cache daily arrays. Target labels stay separate from model features."""
    root = Path(cfg.output).parent / 'graphs'
    rows = []
    for split in ('train', 'val'):
        dates = graph_dates(cfg, split)
        if not dates:
            raise ValueError(f'No {split} transactions in configured dates')
        for date in dates:
            folder = root / date
            if not (folder / 'complete.json').exists():
                day = graph36(cfg, date)
                folder.mkdir(parents=True, exist_ok=True)
                for name, value in vars(day).items():
                    if name != 'day':
                        with (folder / f'{name}.tmp').open('wb') as stream:
                            np.save(stream, value, allow_pickle=False)
                        os.replace(folder / f'{name}.tmp', folder / f'{name}.npy')
                data.json_write(folder / 'complete.json', {'date': date, 'targets': len(day.target_ids)})
            day = read_graph(cfg, date)
            if day.node_x.shape[1] != 19 or day.target_x.shape[1] != 17:
                raise ValueError('Graph feature dimensions differ from schema')
            rows.append({'split': split, 'date': date, 'nodes': len(day.node_ids),
                         'context_edges': len(day.context_src), 'targets': len(day.target_ids),
                         'positives': int(day.labels.sum())})
            print('Graph ready:', split, date, flush=True)
    frame = pd.DataFrame(rows)
    frame.to_csv(root / 'index.csv', index=False)
    return frame

## `read_graph`

날짜 하나의 저장 그래프를 메모리 매핑으로 읽어 전체 기간을 한꺼번에 RAM에 올리지 않습니다.

**입력:** `cfg`, `date`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `data.DayData(day=date, **values)`

In [ ]:
def read_graph(cfg, date):
    """Read only one day's arrays at a time using memory mapping."""
    folder = Path(cfg.output).parent / 'graphs' / date
    if not (folder / 'complete.json').exists():
        raise FileNotFoundError(f'Generate graph first: {date}')
    values = {name: np.load(folder / f'{name}.npy', mmap_mode='r', allow_pickle=False)
              for name in data.DayData.__annotations__ if name != 'day'}
    return data.DayData(day=date, **values)

## 다음 단계에 연결

실행한 코드 셀을 내부 모듈로 저장합니다. 내용을 수정했다면 이 셀을 다시 실행하고 통합 노트북의 커널을 재시작하세요.

In [ ]:
export_cells('20_그래프생성_함수설명.ipynb', 'pipeline')